# 02 — Feature Engineering & Scheme Detection

Computes offensive/defensive metrics, builds roster summary features,
derives continuous style scores, and provides **interactive sliders**
to tune scheme-detection thresholds.


In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.features import build_team_metrics, build_roster_features, compute_ivy_record
from src.scheme_detector import compute_style_scores, apply_labels, build_scheme_widget

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## Load raw data

In [ ]:
stats_df    = pd.read_csv('../data/raw/team_stats/team_stats_raw.csv')
rosters_df  = pd.read_csv('../data/raw/rosters/rosters_raw.csv')
schedules_df = pd.read_csv('../data/raw/schedules/schedules_raw.csv')

print('Stats:', stats_df.shape, '| Rosters:', rosters_df.shape, '| Schedules:', schedules_df.shape)

## Team-level metrics

In [ ]:
metrics_df = build_team_metrics(stats_df)
print('New metric columns:', [c for c in metrics_df.columns if c not in stats_df.columns])
metrics_df[['school','year','yards_per_play','pass_rate','havoc_rate','pace_proxy']].head(10)

## Roster features

In [ ]:
roster_feats = build_roster_features(rosters_df)
print('Roster feature columns:', roster_feats.columns.tolist())
roster_feats.head()

## Ivy win record

In [ ]:
ivy_record = compute_ivy_record(schedules_df)
ivy_record.head(10)

## Merge into master dataset

In [ ]:
master = (
    metrics_df
    .merge(roster_feats, on=['school','year'], how='left')
    .merge(ivy_record,   on=['school','year'], how='left')
)
print('Master shape:', master.shape)
master[['school','year','ivy_win_pct','yards_per_play','pass_rate','havoc_rate']].head()

## Style scores — continuous [0,1] dimensions

In [ ]:
style_df = compute_style_scores(master)
style_df.head(10)

In [ ]:
# Heatmap of style scores per school-year
style_cols = ['pass_tendency','tempo','spread_factor','explosiveness','aggression','front_heaviness']
available = [c for c in style_cols if c in style_df.columns]

pivot = style_df.dropna(subset=available, how='all').pivot_table(
    index='school', columns='year', values='pass_tendency'
)
fig, ax = plt.subplots(figsize=(14,5))
sns.heatmap(pivot, cmap='RdYlGn', center=0.5, ax=ax, linewidths=0.3, cbar_kws={'label':'Pass tendency (0=run, 1=pass)'})
ax.set_title('Pass Tendency by School & Year')
plt.tight_layout()
plt.savefig('../data/processed/pass_tendency_heatmap.png', dpi=150)
plt.show()

## Interactive Scheme Detection Sliders
Drag the sliders to adjust detection thresholds.
The table updates live showing the resulting scheme labels for every team-year.

In [ ]:
build_scheme_widget(style_df)

## Export with default thresholds

In [ ]:
labeled = apply_labels(style_df)
master_labeled = master.merge(labeled[['school','year','off_scheme','def_scheme']], on=['school','year'], how='left')
master_labeled.to_csv('../data/processed/master_labeled.csv', index=False)
print('Saved master_labeled.csv —', master_labeled.shape)
master_labeled[['school','year','off_scheme','def_scheme','ivy_win_pct']].head(12)